# LLM Tool-Calling Evaluation Pipeline on Modal

This notebook evaluates LLM tool-calling capabilities on the ToolJSON benchmark (2,067 samples).

## Setup Instructions
1. Upload this notebook to Modal (or any Jupyter environment)
2. Update your API credentials in the config cell below
3. Run all cells

## 1. Install Dependencies

In [ ]:
!pip install -q openai pyyaml pandas tqdm nltk rouge-score
!python -c "import nltk; nltk.download('punkt')" 2>/dev/null || true

## 2. Configuration

**Important:** Update your API credentials below

In [ ]:
# ==================== CONFIG ====================
# Update these with your Modal API credentials

API_BASE = "https://YOUR_MODAL_URL/v1"  # <-- YOUR MODAL ENDPOINT
API_KEY = "sk-xxx"  # <-- YOUR MODAL API KEY
MODEL_NAME = "gemma-4-31b-it"  # Model name on Modal

# Evaluation settings
LIMIT = 100  # Number of samples to evaluate (set to None for full dataset)
WORKERS = 10  # Concurrent workers

# ==================== END CONFIG ====================

import os
import yaml
import json
import time
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Create config.yaml dynamically
config = {
    "agents": {
        MODEL_NAME: {
            "model": MODEL_NAME,
            "api_base": API_BASE,
            "api_key": API_KEY,
            "temperature": 0.0,
            "max_steps": 16,
            "timeout": 30.0
        }
    }
}

# Save config to file
with open("config.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config created for model: {MODEL_NAME}")
print(f"API Base: {API_BASE}")

## 3. Download ToolJSON Dataset

Extract the `toolJSON_full_2067samples.zip` file to get the data.

In [ ]:
# Option 1: If you have the zip file, unzip it here
# !unzip -q toolJSON_full_2067samples.zip

# Option 2: Download from a URL (if hosted)
# !wget -q https://your-url/toolJSON_full_2067samples.zip && !unzip -q toolJSON_full_2067samples.zip

# For this notebook, we'll assume data is in ./Data/toolJSONprocessing/
DATA_PATH = "./Data/toolJSONprocessing/generate_qa_pairs/data/qa_pairs"

import os
if os.path.exists(DATA_PATH):
    files = [f for f in os.listdir(DATA_PATH) if f.endswith('_qa_pairs.json')]
    print(f"Found {len(files)} QA pair files:")
    total = 0
    for f in files:
        with open(os.path.join(DATA_PATH, f)) as fp:
            count = len(json.load(fp))
        print(f"  {f}: {count} samples")
        total += count
    print(f"\nTotal: {total} samples")
else:
    print(f"Data path not found: {DATA_PATH}")
    print("Please upload the toolJSON_full_2067samples.zip file and extract it.")

## 4. Define LLM Client and Metrics

In [ ]:
import re
import json
from openai import OpenAI
import httpx

class LLMClient:
    def __init__(self, model_name, api_base, api_key):
        self.model_name = model_name
        self.client = OpenAI(
            api_key=api_key,
            base_url=api_base,
            timeout=httpx.Timeout(30.0, connect=10.0),
            max_retries=0
        )
    
    def generate(self, messages, stop=None, max_retries=2):
        if stop is None:
            stop = ["\nObservation:", "Observation:"]
        
        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=messages,
                    temperature=0.0,
                    max_tokens=1024,
                    stop=stop
                )
                return response.choices[0].message.content or ""
            except Exception as e:
                if attempt == max_retries - 1:
                    return f"__ERROR__: {str(e)}"
                time.sleep(2 ** attempt)


class Metrics:
    @staticmethod
    def normalize_text(text):
        if not isinstance(text, str):
            text = str(text)
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        return ' '.join(text.split())
    
    @staticmethod
    def _extract_final_answer(text):
        if not isinstance(text, str):
            return ""
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        if lines:
            return lines[-1]
        return ""
    
    @staticmethod
    def exact_match(prediction, truth):
        pred_norm = Metrics.normalize_text(prediction)
        truth_norm = Metrics.normalize_text(truth)
        
        if pred_norm == truth_norm:
            return 1
        
        # Try final answer extraction
        pred_answer = Metrics._extract_final_answer(prediction)
        if pred_answer:
            pred_answer_norm = Metrics.normalize_text(pred_answer)
            if pred_answer_norm == truth_norm:
                return 1
        
        return 0
    
    @staticmethod
    def contains(prediction, truth):
        pred_norm = Metrics.normalize_text(prediction)
        truth_norm = Metrics.normalize_text(truth)
        return int(truth_norm in pred_norm)
    
    @staticmethod
    def f1_score(prediction, truth):
        pred_tokens = Metrics.normalize_text(prediction).split()
        truth_tokens = Metrics.normalize_text(truth).split()
        
        if len(pred_tokens) == 0 or len(truth_tokens) == 0:
            return int(pred_tokens == truth_tokens)
        
        from collections import Counter
        common_tokens = Counter(pred_tokens) & Counter(truth_tokens)
        num_same = sum(common_tokens.values())
        
        if num_same == 0:
            return 0.0
        
        precision = 1.0 * num_same / len(pred_tokens)
        recall = 1.0 * num_same / len(truth_tokens)
        f1 = (2 * precision * recall) / (precision + recall)
        return f1

print("LLM Client and Metrics classes defined")

## 5. Define ToolJSON Data Loader

In [ ]:
import os
import json

class ToolJSONLoader:
    """Loads ToolJSON QA pairs data"""
    
    def __init__(self, data_path, base_repo_dir="./Data/toolJSONprocessing"):
        self.data_path = data_path
        self.base_repo_dir = base_repo_dir
        self.data = self.load_data()
    
    def load_data(self):
        data = []
        if os.path.isdir(self.data_path):
            for filename in os.listdir(self.data_path):
                if filename.endswith(".json"):
                    file_path = os.path.join(self.data_path, filename)
                    with open(file_path, 'r', encoding='utf-8') as f:
                        file_data = json.load(f)
                        if isinstance(file_data, list):
                            data.extend(file_data)
        return data
    
    def format_prompt(self, sample):
        """Format sample into messages for LLM"""
        question = sample.get("question", "")
        api_query = sample.get("api_query", "")
        
        # Load and process JSON data
        api_response_path = sample.get("api_response_path", "")
        api_res_full_path = os.path.join(self.base_repo_dir, "generate_qa_pairs", "data", api_response_path)
        
        json_content = ""
        try:
            with open(api_res_full_path, 'r', encoding='utf-8') as f:
                raw_json = f.read()
                # For large JSON, we might need to prune (simplified here)
                if len(raw_json) > 50000:
                    json_content = raw_json[:50000] + "\n... (truncated for context)"
                else:
                    json_content = raw_json
        except Exception as e:
            json_content = f"<Error loading JSON: {e}>"
        
        prompt = "Analyze the following JSON output to answer the user's question.\n"
        if api_query:
            prompt += f"API Query Context: {api_query}\n"
        prompt += f"JSON Output:\n{json_content}\n\n"
        prompt += f"Question:\n{question}\n"
        prompt += "Answer directly based on the JSON content."
        
        return [{"role": "user", "content": prompt}]
    
    def get_ground_truth(self, sample):
        return sample.get("gold_answer", "")
    
    def get_question(self, sample):
        return sample.get("question", "Solve the task based on the prompt.")

print("ToolJSONLoader class defined")

## 6. Run Evaluation

In [ ]:
# Initialize
loader = ToolJSONLoader(DATA_PATH)
client = LLMClient(MODEL_NAME, API_BASE, API_KEY)

# Apply limit if specified
data = loader.data[:LIMIT] if LIMIT else loader.data

print(f"Evaluating {len(data)} samples with model: {MODEL_NAME}")
print(f"Workers: {WORKERS}, Limit: {LIMIT}")
print()

results = []
trace_log = []

def process_sample(idx, sample):
    messages = loader.format_prompt(sample)
    ground_truth = loader.get_ground_truth(sample)
    question = loader.get_question(sample)
    
    start = time.time()
    
    try:
        # Generate prediction
        prediction = client.generate(messages)
        elapsed = time.time() - start
        
        if prediction.startswith("__ERROR__"):
            return {
                "sample_id": idx,
                "question": question,
                "error": prediction,
                "prediction": "",
                "ground_truth": ground_truth,
                "exact_match": 0,
                "contains": 0,
                "f1_score": 0
            }
        
        # Compute metrics
        em = Metrics.exact_match(prediction, ground_truth)
        contains = Metrics.contains(prediction, ground_truth)
        f1 = Metrics.f1_score(prediction, ground_truth)
        
        return {
            "sample_id": idx,
            "question": question,
            "prediction": prediction,
            "ground_truth": ground_truth,
            "exact_match": em,
            "contains": contains,
            "f1_score": f1,
            "time_seconds": elapsed
        }
        
    except Exception as e:
        elapsed = time.time() - start
        return {
            "sample_id": idx,
            "question": question,
            "error": str(e),
            "prediction": "",
            "ground_truth": ground_truth,
            "exact_match": 0,
            "contains": 0,
            "f1_score": 0
        }

# Run evaluation with ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_sample, i, s): i for i, s in enumerate(data)}
    
    for future in tqdm(as_completed(futures), total=len(data), desc="Evaluating"):
        try:
            result = future.result(timeout=60)
            results.append(result)
        except Exception as e:
            pass

# Sort by sample_id
results.sort(key=lambda x: x["sample_id"])

print(f"\nCompleted {len(results)} samples")

## 7. Display Results

In [ ]:
# Calculate summary metrics
from statistics import mean

valid_results = [r for r in results if "error" not in r]
error_results = [r for r in results if "error" in r]

summary = {
    "total_samples": len(results),
    "successful": len(valid_results),
    "errors": len(error_results),
    "exact_match": mean([r["exact_match"] for r in valid_results]) if valid_results else 0,
    "contains": mean([r["contains"] for r in valid_results]) if valid_results else 0,
    "f1_score": mean([r["f1_score"] for r in valid_results]) if valid_results else 0,
    "avg_time": mean([r.get("time_seconds", 0) for r in valid_results]) if valid_results else 0
}

print("=" * 50)
print(f"EVALUATION RESULTS: {MODEL_NAME}")
print("=" * 50)
print(f"Total Samples:     {summary['total_samples']}")
print(f"Successful:        {summary['successful']}")
print(f"Errors:            {summary['errors']}")
print(f"\nMetrics:")
print(f"  Exact Match:     {summary['exact_match']:.4f}")
print(f"  Contains:        {summary['contains']:.4f}")
print(f"  F1 Score:        {summary['f1_score']:.4f}")
print(f"  Avg Time:        {summary['avg_time']:.2f}s")
print("=" * 50)

## 8. Save Results

In [ ]:
# Save detailed results as JSON
output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save detailed results
detailed_file = f"{output_dir}/tooljson_{MODEL_NAME}_{timestamp}_detailed.json"
with open(detailed_file, "w", encoding="utf-8") as f:
    json.dump({
        "summary": summary,
        "details": results
    }, f, indent=2, ensure_ascii=False)

# Save as CSV
import pandas as pd
df = pd.DataFrame(results)
csv_file = f"{output_dir}/tooljson_{MODEL_NAME}_{timestamp}_detailed.csv"
df.to_csv(csv_file, index=False, encoding="utf-8")

print(f"\nResults saved:")
print(f"  JSON: {detailed_file}")
print(f"  CSV:  {csv_file}")

# Display first 10 results as preview
print("\nPreview (first 10 results):")
display(df.head(10))

## 9. (Optional) Download Results

If running on Modal/colab, you can download the results files.

In [ ]:
# List all result files for download
import os

if os.path.exists(output_dir):
    print(f"Files in {output_dir}/:")
    for f in sorted(os.listdir(output_dir)):
        fpath = os.path.join(output_dir, f)
        size = os.path.getsize(fpath)
        print(f"  {f} ({size:,} bytes)")
    
    # Create a download link (works in Jupyter)
    from IPython.display import FileLink
    print("\nDownload links:")
    for f in sorted(os.listdir(output_dir)):
        if f.endswith('.json') or f.endswith('.csv'):
            display(FileLink(os.path.join(output_dir, f)))